# Positive Definite Matrices - Examples

Demonstrations of positive definite matrix concepts and their applications.

## Contents
1. Quadratic Form Definition
2. Eigenvalue Test
3. Cholesky Decomposition
4. Solving Linear Systems
5. Covariance Matrices
6. Kernel Matrices
7. Convexity Connection
8. Regularization
9. Nearest PD Matrix
10. Mahalanobis Distance
11. Multiple PD Tests
12. Visualizations

In [ ]:
import numpy as np
from numpy.linalg import eigvalsh, eigh, cholesky, inv, det
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

np.set_printoptions(precision=4, suppress=True)

## 1. Quadratic Form Definition

A symmetric matrix $A$ is **positive definite (PD)** if:

$$\mathbf{x}^T A \mathbf{x} > 0 \quad \text{for all } \mathbf{x} \neq \mathbf{0}$$

**Notation**:
- Positive definite: $A \succ 0$
- Positive semi-definite: $A \succeq 0$ (allows equality)

In [ ]:
# Positive definite matrix
A = np.array([[2, 1],
              [1, 2]])

print(f"A = \n{A}")
print("\nTesting x^T A x for various x:")

test_vectors = [
    np.array([1, 0]),
    np.array([0, 1]),
    np.array([1, 1]),
    np.array([1, -1]),
    np.array([2, 3])
]

for x in test_vectors:
    quadratic = x.T @ A @ x
    print(f"  x = {x}: x^T A x = {quadratic}")

print("\nAll values > 0 → A is positive definite!")

In [ ]:
# Compare with indefinite matrix
B = np.array([[1, 2],
              [2, 1]])

print(f"B = \n{B}")
print("\nTesting x^T B x:")

for x in test_vectors:
    quadratic = x.T @ B @ x
    sign = ">" if quadratic > 0 else ("<" if quadratic < 0 else "=")
    print(f"  x = {x}: x^T B x = {quadratic:3d} {sign} 0")

print("\nSome positive, some negative → B is INDEFINITE!")

## 2. Eigenvalue Test

**Theorem**: A symmetric matrix $A$ is:
- **PD** iff all eigenvalues $\lambda_i > 0$
- **PSD** iff all eigenvalues $\lambda_i \geq 0$
- **Indefinite** if eigenvalues have mixed signs

In [ ]:
matrices = {
    'A (PD)': np.array([[3, 1], [1, 2]]),
    'B (PSD)': np.array([[1, 1], [1, 1]]),
    'C (Indefinite)': np.array([[1, 2], [2, 1]])
}

for name, M in matrices.items():
    eigenvalues = eigvalsh(M)  # eigvalsh for symmetric matrices
    print(f"\n{name}:")
    print(f"  Matrix: {M.tolist()}")
    print(f"  Eigenvalues: {np.round(eigenvalues, 4)}")
    
    if all(eigenvalues > 1e-10):
        print("  → Positive Definite (all λ > 0)")
    elif all(eigenvalues >= -1e-10):
        print("  → Positive Semi-Definite (all λ ≥ 0)")
    else:
        print("  → Indefinite (mixed signs)")

## 3. Cholesky Decomposition

For positive definite $A$:

$$A = LL^T$$

where $L$ is lower triangular with positive diagonal.

**Benefits**:
- More efficient than LU: $\frac{n^3}{3}$ vs $\frac{2n^3}{3}$ operations
- No pivoting needed
- Numerically stable

In [ ]:
A = np.array([[4, 2, 2],
              [2, 5, 1],
              [2, 1, 6]], dtype=float)

print(f"A = \n{A}")
print(f"\nEigenvalues: {np.round(eigvalsh(A), 4)}")
print("All positive → A is PD → Cholesky exists")

In [ ]:
# Manual Cholesky
print("--- Manual Cholesky ---")
n = 3
L = np.zeros((n, n))

for j in range(n):
    # Diagonal element
    sum_sq = sum(L[j, k]**2 for k in range(j))
    L[j, j] = np.sqrt(A[j, j] - sum_sq)
    print(f"L[{j},{j}] = √(A[{j},{j}] - Σ L[{j},k]²) = √({A[j,j]:.0f} - {sum_sq:.2f}) = {L[j,j]:.4f}")
    
    # Below diagonal
    for i in range(j + 1, n):
        sum_prod = sum(L[i, k] * L[j, k] for k in range(j))
        L[i, j] = (A[i, j] - sum_prod) / L[j, j]

print(f"\nL (lower triangular) = \n{np.round(L, 4)}")

# Verify
print(f"\nVerification: L @ L^T = \n{np.round(L @ L.T, 4)}")
print(f"Matches A? {np.allclose(L @ L.T, A)}")

In [ ]:
# NumPy Cholesky
print("--- NumPy Cholesky ---")
L_np = cholesky(A)
print(f"L = \n{np.round(L_np, 4)}")

## 4. Solving Linear Systems with Cholesky

For $Ax = b$ where $A$ is PD:

1. Cholesky: $A = LL^T$
2. Forward solve: $Ly = b$
3. Back solve: $L^T x = y$

More efficient and stable than general methods!

In [ ]:
A = np.array([[4, 2],
              [2, 5]], dtype=float)
b = np.array([8, 11], dtype=float)

print(f"A = \n{A}")
print(f"b = {b}")

# Step 1: Cholesky
L = cholesky(A)
print(f"\nStep 1: A = L L^T")
print(f"L = \n{np.round(L, 4)}")

# Step 2: Forward substitution
y = np.zeros(2)
y[0] = b[0] / L[0, 0]
y[1] = (b[1] - L[1, 0] * y[0]) / L[1, 1]
print(f"\nStep 2: Solve Ly = b (forward)")
print(f"y = {np.round(y, 4)}")

# Step 3: Back substitution
LT = L.T
x = np.zeros(2)
x[1] = y[1] / LT[1, 1]
x[0] = (y[0] - LT[0, 1] * x[1]) / LT[0, 0]
print(f"\nStep 3: Solve L^T x = y (backward)")
print(f"x = {np.round(x, 4)}")

# Verify
print(f"\nVerification: A @ x = {np.round(A @ x, 4)} = b ✓")

## 5. Covariance Matrices

**Key Property**: Covariance matrices are always positive semi-definite!

$$\Sigma = \mathbb{E}[(\mathbf{X} - \mu)(\mathbf{X} - \mu)^T]$$

**Proof**: For any $\mathbf{a}$:
$$\mathbf{a}^T \Sigma \mathbf{a} = \text{Var}(\mathbf{a}^T \mathbf{X}) \geq 0$$

In [ ]:
np.random.seed(42)

# Generate correlated data
n = 1000
X = np.random.randn(n, 3)
# Add correlation
X[:, 1] = X[:, 0] * 0.8 + X[:, 1] * 0.2
X[:, 2] = X[:, 0] * 0.5 + X[:, 2] * 0.5

# Compute covariance
X_centered = X - X.mean(axis=0)
Sigma = X_centered.T @ X_centered / (n - 1)

print(f"Covariance matrix Σ:\n{np.round(Sigma, 4)}")

eigenvalues = eigvalsh(Sigma)
print(f"\nEigenvalues: {np.round(eigenvalues, 4)}")
print(f"All ≥ 0? {all(eigenvalues >= -1e-10)}")
print("→ Σ is positive semi-definite!")

In [ ]:
# Why is it PSD? Variance interpretation
print("--- Why covariance is PSD ---")
a = np.array([1, 2, 1])
var_linear_combo = a.T @ Sigma @ a
print(f"For any a = {a}:")
print(f"a^T Σ a = Var(a^T X) = {var_linear_combo:.4f}")
print("Variance is always ≥ 0!")

## 6. Kernel Matrices

For kernel methods (SVM, GP), the kernel matrix must be PSD:

$$K_{ij} = K(\mathbf{x}_i, \mathbf{x}_j)$$

Common valid kernels:
- Linear: $K(\mathbf{x}, \mathbf{y}) = \mathbf{x}^T\mathbf{y}$
- RBF: $K(\mathbf{x}, \mathbf{y}) = \exp(-\gamma\|\mathbf{x} - \mathbf{y}\|^2)$
- Polynomial: $K(\mathbf{x}, \mathbf{y}) = (\mathbf{x}^T\mathbf{y} + c)^d$

In [ ]:
# Data points
X = np.array([[0], [1], [2], [3]], dtype=float)
print(f"Data points: {X.flatten()}")

# RBF kernel
gamma = 0.5
n = len(X)
K = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        K[i, j] = np.exp(-gamma * (X[i] - X[j])**2)

print(f"\nRBF Kernel matrix (γ = {gamma}):")
print(np.round(K, 4))

eigenvalues = eigvalsh(K)
print(f"\nEigenvalues: {np.round(eigenvalues, 4)}")
print(f"All ≥ 0? {all(eigenvalues >= -1e-10)}")
print("→ Kernel matrix is PSD!")

In [ ]:
# Linear kernel
K_linear = X @ X.T
print(f"Linear kernel K = X @ X^T:")
print(K_linear)
print(f"Eigenvalues: {np.round(eigvalsh(K_linear), 4)}")
print("Also PSD (since K = X X^T = B^T B for B = X^T)")

## 7. Connection to Convexity

For quadratic function $f(\mathbf{x}) = \frac{1}{2}\mathbf{x}^T A \mathbf{x} - \mathbf{b}^T\mathbf{x}$:

- **Hessian**: $\nabla^2 f = A$
- **$A \succ 0$** → $f$ is strictly convex → unique global minimum
- **$A$ indefinite** → $f$ has a saddle point

In [ ]:
print("Quadratic function: f(x) = 0.5 x^T A x - b^T x")
print("Gradient: ∇f = Ax - b")
print("Hessian: ∇²f = A")

# Convex case
A_convex = np.array([[2, 1], [1, 2]])
b = np.array([1, 1])

print(f"\n--- Case 1: Convex ---")
print(f"A = \n{A_convex}")
print(f"Eigenvalues: {eigvalsh(A_convex)}")
print("All positive → f is strictly convex")
print(f"Unique minimum at x* = A⁻¹b = {np.round(inv(A_convex) @ b, 4)}")

# Non-convex case (saddle)
A_nonconvex = np.array([[1, 2], [2, 1]])

print(f"\n--- Case 2: Saddle Point ---")
print(f"A = \n{A_nonconvex}")
print(f"Eigenvalues: {eigvalsh(A_nonconvex)}")
print("Mixed signs → f has a saddle point (not convex)")

In [ ]:
# Visualize convex vs saddle
fig = plt.figure(figsize=(12, 5))

x = np.linspace(-3, 3, 50)
y = np.linspace(-3, 3, 50)
X_grid, Y_grid = np.meshgrid(x, y)

# Convex: f(x) = 0.5 x^T A x
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
Z1 = 0.5 * (A_convex[0,0]*X_grid**2 + 2*A_convex[0,1]*X_grid*Y_grid + A_convex[1,1]*Y_grid**2)
ax1.plot_surface(X_grid, Y_grid, Z1, cmap='viridis', alpha=0.8)
ax1.set_xlabel('x₁'); ax1.set_ylabel('x₂'); ax1.set_zlabel('f(x)')
ax1.set_title('Positive Definite A\n(Bowl - Convex)')

# Saddle
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
Z2 = 0.5 * (A_nonconvex[0,0]*X_grid**2 + 2*A_nonconvex[0,1]*X_grid*Y_grid + A_nonconvex[1,1]*Y_grid**2)
ax2.plot_surface(X_grid, Y_grid, Z2, cmap='coolwarm', alpha=0.8)
ax2.set_xlabel('x₁'); ax2.set_ylabel('x₂'); ax2.set_zlabel('f(x)')
ax2.set_title('Indefinite A\n(Saddle Point)')

plt.tight_layout()
plt.show()

## 8. Regularization Makes PD

Adding $\lambda I$ to a singular or PSD matrix makes it PD:

$$A + \lambda I \succ 0 \quad \text{for } \lambda > -\lambda_{\min}(A)$$

In [ ]:
# Singular matrix (rank deficient)
X = np.array([[1, 2],
              [2, 4],
              [3, 6]])  # Columns are linearly dependent

XTX = X.T @ X
print(f"X = \n{X}")
print(f"\nX^T X = \n{XTX}")
print(f"Eigenvalues: {np.round(eigvalsh(XTX), 4)}")
print("One eigenvalue ≈ 0 → Singular!")

In [ ]:
# Add regularization
print("--- Adding regularization λI ---\n")

for lam in [0.1, 1.0, 10.0]:
    XTX_reg = XTX + lam * np.eye(2)
    eigenvalues = eigvalsh(XTX_reg)
    print(f"λ = {lam:4.1f}: eigenvalues = {np.round(eigenvalues, 4)}")
    print(f"         All > 0? {all(eigenvalues > 0)} → Now invertible!")
    print()

## 9. Finding Nearest PD Matrix

Estimated covariance matrices may not be PD due to:
- Numerical errors
- More variables than samples
- Missing data

**Solution**: Project negative eigenvalues to small positive values.

In [ ]:
# Correlation matrix that's not quite valid
A = np.array([[1.0, 0.9, 0.8],
              [0.9, 1.0, 0.9],
              [0.8, 0.9, 1.0]])

# Make it slightly non-PD
A[0, 2] = 0.85
A[2, 0] = 0.85

print(f"A = \n{A}")
eigenvalues, eigenvectors = eigh(A)
print(f"Eigenvalues: {np.round(eigenvalues, 6)}")

if any(eigenvalues < 0):
    print("Some eigenvalues negative → Not PD!")

In [ ]:
# Fix by projecting to PD cone
def nearest_pd(A, epsilon=1e-6):
    """Project to nearest positive definite matrix."""
    eigenvalues, eigenvectors = eigh(A)
    eigenvalues_fixed = np.maximum(eigenvalues, epsilon)
    return eigenvectors @ np.diag(eigenvalues_fixed) @ eigenvectors.T

A_pd = nearest_pd(A)

print(f"\nNearest PD matrix:")
print(f"A_pd = \n{np.round(A_pd, 4)}")

print(f"\nNew eigenvalues: {np.round(eigvalsh(A_pd), 6)}")

# Verify
try:
    cholesky(A_pd)
    print("Cholesky successful → A_pd is PD!")
except:
    print("Still not PD")

## 10. Mahalanobis Distance

The Mahalanobis distance accounts for correlations:

$$d_M(\mathbf{x}, \mathbf{y}) = \sqrt{(\mathbf{x}-\mathbf{y})^T\Sigma^{-1}(\mathbf{x}-\mathbf{y})}$$

Requires $\Sigma \succ 0$ for valid distance metric.

In [ ]:
# Covariance with correlation
Sigma = np.array([[2, 1],
                  [1, 2]])
mu = np.array([0, 0])

print(f"Σ = \n{Sigma}")
print(f"μ = {mu}")

Sigma_inv = inv(Sigma)
print(f"\nΣ⁻¹ = \n{np.round(Sigma_inv, 4)}")

def mahalanobis(x, mu, Sigma_inv):
    diff = x - mu
    return np.sqrt(diff.T @ Sigma_inv @ diff)

def euclidean(x, mu):
    diff = x - mu
    return np.sqrt(diff.T @ diff)

In [ ]:
# Compare distances
points = [
    np.array([1, 0]),
    np.array([0, 1]),
    np.array([1, 1]),
    np.array([2, 0])
]

print("Compare Euclidean vs Mahalanobis distances:")
for x in points:
    d_eucl = euclidean(x, mu)
    d_maha = mahalanobis(x, mu, Sigma_inv)
    print(f"  x = {x}: Euclidean = {d_eucl:.3f}, Mahalanobis = {d_maha:.3f}")

print("\nMahalanobis accounts for correlation structure!")
print("Points along high-variance directions are 'closer'.")

In [ ]:
# Visualize Euclidean vs Mahalanobis "unit circles"
theta = np.linspace(0, 2*np.pi, 100)

# Euclidean unit circle
eucl_circle = np.array([np.cos(theta), np.sin(theta)])

# Mahalanobis unit circle: d_M = 1
# Need to transform by sqrt(Sigma)
sqrt_Sigma = np.linalg.cholesky(Sigma)
maha_circle = sqrt_Sigma @ eucl_circle

plt.figure(figsize=(8, 6))
plt.plot(eucl_circle[0], eucl_circle[1], 'b-', linewidth=2, label='Euclidean d=1')
plt.plot(maha_circle[0], maha_circle[1], 'r-', linewidth=2, label='Mahalanobis d=1')
plt.scatter([0], [0], c='black', s=100, zorder=5)
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.legend()
plt.title('Euclidean vs Mahalanobis Distance (d=1 contours)')
plt.xlabel('x₁'); plt.ylabel('x₂')
plt.show()

## 11. Multiple PD Tests

Several equivalent ways to test positive definiteness:

1. **Symmetry**: $A = A^T$
2. **Eigenvalue test**: All $\lambda_i > 0$
3. **Sylvester's criterion**: All leading principal minors > 0
4. **Cholesky test**: Decomposition exists

In [ ]:
A = np.array([[2, -1, 0],
              [-1, 2, -1],
              [0, -1, 2]])

print(f"A = \n{A}")

# Test 1: Symmetry
print("\nTest 1: Symmetry")
print(f"  A = A^T? {np.allclose(A, A.T)} ✓")

# Test 2: Eigenvalues
print("\nTest 2: Eigenvalues")
eigenvalues = eigvalsh(A)
print(f"  Eigenvalues: {np.round(eigenvalues, 4)}")
print(f"  All > 0? {all(eigenvalues > 0)} ✓")

# Test 3: Leading principal minors (Sylvester)
print("\nTest 3: Leading Principal Minors")
print(f"  a₁₁ = {A[0,0]} > 0? {A[0,0] > 0} ✓")
minor2 = det(A[:2, :2])
print(f"  det(A[:2,:2]) = {minor2:.4f} > 0? {minor2 > 0} ✓")
minor3 = det(A)
print(f"  det(A) = {minor3:.4f} > 0? {minor3 > 0} ✓")

# Test 4: Cholesky
print("\nTest 4: Cholesky Decomposition")
try:
    L = cholesky(A)
    print(f"  Cholesky exists! ✓")
except:
    print("  Cholesky failed → Not PD")

print("\n✓ A is positive definite!")

## 12. Visualizations: Quadratic Forms

In [ ]:
# Visualize different types of matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

matrices = [
    ('Positive Definite', np.array([[2, 0.5], [0.5, 1]])),
    ('Positive Semi-Definite', np.array([[1, 1], [1, 1]])),
    ('Indefinite', np.array([[1, 2], [2, 1]]))
]

x = np.linspace(-2, 2, 100)
y = np.linspace(-2, 2, 100)
X_grid, Y_grid = np.meshgrid(x, y)

for ax, (name, A) in zip(axes, matrices):
    # f(x) = x^T A x
    Z = A[0,0]*X_grid**2 + 2*A[0,1]*X_grid*Y_grid + A[1,1]*Y_grid**2
    
    # Contour plot
    contour = ax.contourf(X_grid, Y_grid, Z, levels=20, cmap='RdYlBu_r')
    ax.contour(X_grid, Y_grid, Z, levels=20, colors='black', alpha=0.3, linewidths=0.5)
    
    eigenvalues = eigvalsh(A)
    ax.set_title(f'{name}\nλ = {np.round(eigenvalues, 2)}')
    ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
    ax.set_aspect('equal')
    ax.scatter([0], [0], c='white', s=100, zorder=5, edgecolors='black')

plt.tight_layout()
plt.show()

print("PD: Bowl shape (minimum at origin)")
print("PSD: Valley (minimum along a line)")
print("Indefinite: Saddle (min in one direction, max in another)")

## Summary

### Equivalent Definitions of PD

| Test | Condition |
|------|-----------|  
| Quadratic form | $\mathbf{x}^T A \mathbf{x} > 0$ for all $\mathbf{x} \neq \mathbf{0}$ |
| Eigenvalues | All $\lambda_i > 0$ |
| Sylvester | All leading principal minors $> 0$ |
| Cholesky | $A = LL^T$ exists |
| Decomposition | $A = B^TB$ for full-rank $B$ |

### ML Applications

```
Positive Definiteness in ML:
├── Covariance matrices (PSD)
│   └── Gaussian distributions, sampling
├── Kernel matrices (PSD)
│   └── SVM, Gaussian Processes
├── Hessians (PD → convex)
│   └── Newton's method, optimization
├── Regularization (ensures PD)
│   └── Ridge regression, stability
└── Metric learning (PD distance matrix)
    └── Mahalanobis distance
```

### Key Properties

- PD matrices are always invertible
- If $A$ is PD, so is $A^{-1}$
- Sum of PD matrices is PD
- $A^TA$ is always PSD
- Regularization $A + \lambda I$ makes PD for $\lambda > -\lambda_{\min}$